<a href="https://colab.research.google.com/github/Vynxvoid/WorldBank_WebScraping/blob/main/WorldBank_Projects_Web_Scraping.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [34]:
%%shell
wget -q -O - https://dl-ssl.google.com/linux/linux_signing_key.pub | apt-key add -
echo "deb [arch=amd64] http://dl.google.com/linux/chrome/deb/ stable main" >> /etc/apt/sources.list.d/google.list
apt-get update
apt-get install -y google-chrome-stable

pip install webdriver-manager selenium

OK
Hit:1 http://dl.google.com/linux/chrome/deb stable InRelease
Get:2 https://dl.google.com/linux/chrome-stable/deb stable InRelease [1,825 B]
Hit:3 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:4 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:5 https://cli.github.com/packages stable InRelease
Hit:6 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Get:8 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:9 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:12 https://dl.google.com/linux/chrome-stable/deb stable/main amd64 Packages [1,213 B]
Get:13 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [10.1 MB]
Get:14 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [2,985 kB]
Fetche

CalledProcessError: Command '# Install official Google Chrome
wget -q -O - https://dl-ssl.google.com/linux/linux_signing_key.pub | apt-key add -
echo "deb [arch=amd64] http://dl.google.com/linux/chrome/deb/ stable main" >> /etc/apt/sources.list.d/google.list
apt-get update
apt-get install -y google-chrome-stable

# Install webdriver-manager to handle driver matching
pip install webdriver-manager selenium
' died with <Signals.SIGINT: 2>.

In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
import pandas as pd
import time
import os
from concurrent.futures import ThreadPoolExecutor, as_completed

def get_driver():
    options = Options()
    options.add_argument('--headless')
    options.add_argument('--no-sandbox')
    options.add_argument('--disable-dev-shm-usage')
    options.add_experimental_option('excludeSwitches', ['enable-logging'])
    service = Service(ChromeDriverManager().install())
    return webdriver.Chrome(service=service, options=options)

def scrape_contract(contractId):
    driver=get_driver()
    url=f'https://projects.worldbank.org/en/projects-operations/contractoverview/{contractId}'
    projectData={'Contract ID': contractId}

    try:
        driver.get(url)
        time.sleep(2)

        rows=driver.find_elements('css selector', 'ul.row')
        if not rows:
            return None
        foundData=False
        for row in rows:
            items=row.find_elements('tag name', 'li')
            for item in items:
                try:
                    label=item.find_element('tag name', 'label').text.strip().rstrip(':')
                    value=item.find_element('tag name', 'p').text.strip()
                    if label and value:
                        projectData[label]=value
                        foundData=True
                except:
                    continue
        return projectData if foundData else None
    except:
        return None
    finally:
        driver.quit()

maxRecords=1870
numWorkers=4
startId=1880000
allResults=[]
currentBatchStart=startId
batchSize=20

while len(allResults) < maxRecords:
    toCheck=[str(i).zfill(8) for i in range(currentBatchStart, currentBatchStart + batchSize)]

    with ThreadPoolExecutor(max_workers=numWorkers) as executor:
        future_to_id={executor.submit(scrape_contract, cid): cid for cid in toCheck}

        for future in as_completed(future_to_id):
            data=future.result()
            if data:
                allResults.append(data)
                if len(allResults)>=maxRecords:
                    break

    print(f"Progress: {len(allResults)}/{maxRecords} records found. Last ID checked: {toCheck[-1]}")

    if allResults:
        pd.DataFrame(allResults).to_csv('world_bank_contracts.csv',index=False)

    currentBatchStart += batchSize
    if currentBatchStart > startId + 50000:
        print("Checked 50,000 IDs, stopping search.")
        break

print(f"Finished. Total records saved: {len(allResults)}")
df=pd.DataFrame(allResults)
df.to_csv('world_bank_contracts.csv',index=False)

In [35]:
df=pd.read_csv('world_bank_contracts.csv')
for label in df.columns:
  print(label)


Contract ID
Project ID
Project Title
Contract No
Procurement Group
Country
No Objection Date
Team Leader
Procurement Method
Borrower Contract Reference
Signing Date
Total Contract Amount
Procurement Type
Review Type


In [36]:
df.drop(columns=['Procurement Type'],inplace=True)
df['Total Contract Amount']=df['Total Contract Amount'].astype(str).str.replace('US$','',regex=False).str.replace(',','',regex=False)
df['Total Contract Amount']=pd.to_numeric(df['Total Contract Amount'])
df.replace('N/A',pd.NA,inplace=True)
df.fillna(method='ffill',inplace=True)

df.head()

/tmp/ipykernel_2707/3097987206.py:5: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df.fillna(method='ffill',inplace=True)


,Contract ID,Project ID,Project Title,Contract No,Procurement Group,Country,No Objection Date,Team Leader,Procurement Method,Borrower Contract Reference,Signing Date,Total Contract Amount,Review Type
0,1880035,P171316,Growing Up and Learning Together: Comprehensiv...,1880035,Non-consulting Services,El Salvador,"April 9, 2026",Isy Faingold Vigil,Request for Bids,SDO NO. 71/2025 MINEDUCYT-BIRF,"January 14, 2026",134797.70,Post
1,1880036,P173749,Benin Electricity Access Scale-up (BEAS) Project,1880036,Goods,Benin,"April 9, 2026",Justin Marie Bienvenu Beleoken Sanguen,Request for Quotations,BJ-PIU-497127-GO-RFQ,"January 27, 2026",35181.07,Post
2,1880037,P505179,Supporting the Federal Policy for Enhancing Fo...,1880037,Consultant Services,Argentina,"April 9, 2026",Alonso Sanchez,Direct Selection,AR-ME-468827-CS-CDS,"January 1, 2025",2910.14,Post
3,1880038,P162833,Improving the Performance of Non-Criminal Just...,1880038,Consultant Services,Peru,"April 9, 2026",Ruben Leonel Ruano Chinchilla,Individual Consultant Selection,PE-PJ-488144-CS-INDV,"July 25, 2025",11624.36,Post
4,1880039,P173283,Territorial Economic Empowerment for the Indig...,1880039,Non-consulting Services,Ecuador,"April 9, 2026",Kosuke Anan,Request for Quotations,IEPS-PROFECPIAM-002-2025,"April 10, 2025",5000.00,Post


In [37]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1870 entries, 0 to 1869
Data columns (total 13 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   Contract ID                  1870 non-null   int64  
 1   Project ID                   1870 non-null   object 
 2   Project Title                1870 non-null   object 
 3   Contract No                  1870 non-null   int64  
 4   Procurement Group            1870 non-null   object 
 5   Country                      1870 non-null   object 
 6   No Objection Date            1870 non-null   object 
 7   Team Leader                  1870 non-null   object 
 8   Procurement Method           1870 non-null   object 
 9   Borrower Contract Reference  1870 non-null   object 
 10  Signing Date                 1870 non-null   object 
 11  Total Contract Amount        1870 non-null   float64
 12  Review Type                  1870 non-null   object 
dtypes: float64(1), int

In [38]:
df.to_csv('world_bank_contracts.csv',index=False)